# 🐆 Panther AI - Transaction Classifier Training (Colab)

This notebook trains the DistilBERT model for the Panther Transaction Classifier on Google Colab using GPU/TPU.

**Pre-requisites:**
1.  **Dataset**: You need `panther_ground_truth_v3_balanced.parquet`.
2.  **Runtime**: Enable GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Install dependencies
!pip install transformers datasets accelerate scikit-learn torch pandas pyarrow

## 📂 Data Setup

Choose **ONE** of the next two cells to load your data.

In [ ]:
# OPTION A: Mount Google Drive (Best if you have sync working)
from google.colab import drive
try:
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/My Drive/Kalós/martinhijarx/AI Projects/CHRONOS"
    DATASET_PATH = os.path.join(BASE_PATH, "datasets/processed/panther_ground_truth_v3_balanced.parquet")
except:
    print("Drive mount failed or skipped. Use Option B below.")

In [ ]:
# OPTION B: Direct Upload (Use this if Drive is not syncing)
import os
from google.colab import files

if not os.path.exists("panther_ground_truth_v3_balanced.parquet"):
    print("Please upload 'panther_ground_truth_v3_balanced.parquet' from your computer:")
    uploaded = files.upload()
    DATASET_PATH = "panther_ground_truth_v3_balanced.parquet"
else:
    DATASET_PATH = "panther_ground_truth_v3_balanced.parquet"
    print(f"✅ File found in session storage: {DATASET_PATH}")

In [ ]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Constants
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "/content/distilbert_v1_colab"
LOG_DIR = f"{OUTPUT_DIR}/logs"
MAX_LENGTH = 64
BATCH_SIZE = 64  
EPOCHS = 5

print(f"Final Dataset Path: {DATASET_PATH}")

In [ ]:
def normalize_text(text):
    """
    Aggressive text normalization to strip PII and transient IDs using regex.
    """
    text = str(text).upper()
    # Remove hashtags followed by non-whitespace (e.g., #1234, #Store5)
    text = re.sub(r'#\S+', '', text)
    # Remove TXN followed by non-whitespace
    text = re.sub(r'TXN\S+', '', text)
    # Replace sequences of digits with a single space
    text = re.sub(r'\d+', ' ', text)
    # Replace multiple spaces with single space and strip
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

class CustomTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None
        self.focal_loss = FocalLoss(alpha=self.class_weights, gamma=2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None and self.class_weights.device != logits.device:
            self.class_weights = self.class_weights.to(logits.device)
            self.focal_loss.alpha = self.class_weights
        loss = self.focal_loss(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Load and Prep Data
print(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_parquet(DATASET_PATH)

# Normalize
print("Normalizing text...")
df['clean_description'] = df['description'].apply(normalize_text)

# Labels
labels = sorted(df['subcategory'].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)
df['label'] = df['subcategory'].map(label2id)

# Split
print("Splitting dataset 80/10/10...")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Class Weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['label']), y=train_df['label'])

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(train_df[['clean_description', 'label']])
val_dataset = Dataset.from_pandas(val_df[['clean_description', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_description', 'label']])

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    return tokenizer(examples['clean_description'], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
# Initialize Model
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=500, weight_decay=0.01, logging_dir=LOG_DIR, logging_steps=100, eval_strategy="steps", eval_steps=500,
    save_strategy="steps", save_steps=500, load_best_model_at_end=True, metric_for_best_model="accuracy", label_smoothing_factor=0.1,
    report_to="none", fp16=True
)

trainer = CustomTrainer(class_weights=class_weights, model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

print("Starting training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete!")

In [ ]:
# Download model files to local machine
import shutil
from google.colab import files
print("Zipping model files...")
shutil.make_archive("/content/distilbert_v1_colab", 'zip', OUTPUT_DIR)
print("Downloading zip file...")
files.download("/content/distilbert_v1_colab.zip")